# LLaMA-2-7B Observation: DEWA In-Window Rate with and without Activation-Outlier Separation

Observation 2 and Observation 3 for the DEWA motivation. This notebook does **not** compute PPL.

The model runs the baseline **W-BFP4 / A-BFP4 / G16** path (`lm_head` stays FP16), so every
decoder `nn.Linear` sees the same downstream activations. A forward pre-hook replays the Group-16
partial sums of each Linear in true K order on two shadow paths:

- `raw`: the activation is BFP4-quantized with one shared exponent per G16 block. Every partial
  sum enters the accumulator.
- `separated`: activation outliers (tensor-wise `|x| > mu + 3 sigma` of `|x|`) are encoded with
  their own outlier shared exponent (BiE4, uncapped), and normal values with the normal shared
  exponent. Each G16 group then yields a normal partial `P_N` and an outlier partial `P_O`.
  `P_N` enters the accumulator; `P_O` is routed to the FP-Acc in the target design, so it is
  **never dropped**: every nonzero `P_O` counts as in-window.

**Hardware exponent trajectory.** DEW-ACC compares block exponents, not value magnitudes. A group
partial is `P_t = Psum_t * 2^(E_t)` with block exponent `E_t = E_w + E_a` (on the `separated` path,
`E_a` is the normal activation exponent). The accumulator exponent is the largest block exponent
accumulated so far, which is what the DEW-ACC sign mux keeps in its exponent register:

```text
E_acc(t)  = max { E_i : i < t, P_i != 0 }
delta_E   = E_t - E_acc(t)
drop      : delta_E <= -T_drop
replace   : delta_E >= +T_replace
in-window : -T_drop < delta_E < T_replace
```

A drop keeps `E_acc`, a replace sets it to `E_t` (the new maximum), and an in-window add keeps the
larger of the two, so `E_acc(t)` is the running maximum whatever decisions are taken. Each path is
therefore replayed once, only the `delta_E` histogram is stored, and the in-window rate of **any**
`(T_drop, T_replace)` window is derived afterwards with exactly the decisions the hardware makes.
Overflow spills, which reset the accumulator, depend on the accumulator width rather than on the
window and are not modeled.

A zero partial is a hold: it makes no decision and does not update `E_acc`. The first nonzero
partial of a dot product is a load and is excluded from both numerator and denominator.

```text
raw       in-window = in-window decisions / nonzero decisions
separated in-window = (normal in-window decisions + nonzero P_O) / (normal nonzero decisions + nonzero P_O)
```

In [ ]:
%pip install -q "transformers==5.13.1" "datasets==4.0.0" accelerate sentencepiece tqdm

In [ ]:
import gc
import json
import math
import os
import platform
import re
import time
from dataclasses import asdict, dataclass
from getpass import getpass
from pathlib import Path

import datasets
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "meta-llama/Llama-2-7b-hf"
DATASET_ID = "Salesforce/wikitext"
DATASET_CONFIG = "wikitext-2-raw-v1"
SPLIT = "test"
CONTEXT_LENGTH = 2048
STRIDE = 2048
DROP_REMAINDER = True
EVALUATION_PROTOCOL = "non_overlapping_2048_drop_remainder"


@dataclass(frozen=True)
class BFPConfig:
    block_size: int = 16
    shared_exponent_bits: int = 5
    mantissa_bits: int = 3  # BFP4 / BiE4: 1 sign bit + 3 magnitude bits.
    rounding: str = "nearest_even"
    weight_chunk_rows: int = 128
    activation_chunk_rows: int = 256
    quantize_lm_head: bool = False

    def validate(self):
        if self.block_size != 16:
            raise ValueError("This observation is fixed to Group-16.")
        if self.shared_exponent_bits != 5:
            raise ValueError("This observation uses E5 shared exponents.")
        if self.mantissa_bits <= 0:
            raise ValueError("mantissa_bits must be positive.")
        if self.rounding != "nearest_even":
            raise ValueError("This observation uses round-to-nearest-even.")
        if self.quantize_lm_head:
            raise ValueError("lm_head stays FP16 (224 decoder Linear layers).")


@dataclass(frozen=True)
class OutlierConfig:
    method: str = "tensor_mu_plus_k_sigma_of_abs"
    sigma_k: float = 3.0
    max_outliers_per_block: int | None = None  # None: every value above the threshold.

    def validate(self):
        if self.method != "tensor_mu_plus_k_sigma_of_abs":
            raise ValueError("Only the tensor-wise mu + k*sigma rule is implemented.")
        if not math.isfinite(self.sigma_k) or self.sigma_k < 0:
            raise ValueError("sigma_k must be finite and non-negative.")
        if self.max_outliers_per_block is not None and self.max_outliers_per_block <= 0:
            raise ValueError("max_outliers_per_block must be positive or None.")


@dataclass(frozen=True)
class ProfileConfig:
    max_profile_blocks: int = 8          # 2048-token blocks, evenly spaced over the test split
    token_stride: int = 1                # >1 profiles every n-th token to cut cost
    out_subsample: int | None = 1024     # evenly spaced output channels per Linear (None = all)
    token_chunk: int = 128
    max_chunk_elements: int = 1 << 26    # bounds the (group, token, out) working set
    delta_bin_limit: int = 64            # bins -64..64 plus two edge bins (E5+E5 deltas fit in +-62)
    t_drop_values: tuple = tuple(range(6, 13))
    t_replace_values: tuple = tuple(range(1, 13))
    highlight_windows: tuple = ((8, 8), (9, 9), (10, 10), (11, 11), (12, 12), (9, 3))

    def validate(self):
        if min(self.max_profile_blocks, self.token_stride, self.token_chunk,
               self.max_chunk_elements, self.delta_bin_limit) <= 0:
            raise ValueError("Profiling counts and chunk sizes must be positive.")
        if self.out_subsample is not None and self.out_subsample <= 0:
            raise ValueError("out_subsample must be positive or None.")
        windows = [(d, r) for d in self.t_drop_values for r in self.t_replace_values]
        windows += list(self.highlight_windows)
        for t_drop, t_replace in windows:
            if not (1 <= t_drop <= self.delta_bin_limit and 1 <= t_replace <= self.delta_bin_limit):
                raise ValueError("Every T_drop/T_replace must lie in [1, delta_bin_limit].")


BFP = BFPConfig()
OUTLIER = OutlierConfig()
PROFILE = ProfileConfig()
BFP.validate()
OUTLIER.validate()
PROFILE.validate()

_cap_tag = "uncapped" if OUTLIER.max_outliers_per_block is None else f"top{OUTLIER.max_outliers_per_block}"
OUTPUT_PATH = Path("result") / (
    f"llama2-7b-w-bfp{BFP.mantissa_bits + 1}-a-bfp{BFP.mantissa_bits + 1}-vs-bie{BFP.mantissa_bits + 1}-"
    f"{OUTLIER.sigma_k:g}sigma-{_cap_tag}-g{BFP.block_size}-in-window-blkexp-runmax-"
    f"b{PROFILE.max_profile_blocks}-ts{PROFILE.token_stride}-out{PROFILE.out_subsample or 'all'}.json"
)

torch.manual_seed(0)
torch.backends.cuda.matmul.allow_tf32 = False
torch.set_float32_matmul_precision("highest")

print(f"BFP config: {BFP}")
print(f"Outlier config: {OUTLIER}")
print(f"Profile config: {PROFILE}")
print(f"Output: {OUTPUT_PATH.resolve()}")
if not torch.cuda.is_available():
    print("CUDA is unavailable: synthetic checks can run on CPU; the full profiler cannot.")

## BFP4 and BiE4 encodings

Same convention as `05_Top2-ABiE`: the shared exponent of a G16 block is `floor(log2(max|x|))`
clamped to the signed E5 range, the step is `2 ** (shared_exp - (M - 1))`, and mantissas use
round-to-nearest-even in `[-(2**M - 1), 2**M - 1]`.

For the separated path, normal and outlier lanes of a block get their own shared exponent.
`split_bie` returns the dequantized normal lanes and outlier lanes as two tensors, so
`P_N = X_N . W` and `P_O = X_O . W` per group, and `X_N + X_O` is the full BiE4 activation.

`bfp_block_exponent` and `split_bie` also return the per-block shared exponents. The block
exponent of a group partial is `E_w + E_a` (normal `E_a` on the separated path); both operands use
the same mantissa width, so the constant LSB offset `2 (M - 1)` cancels in `delta_E`.

In [ ]:
def _shared_exponent(max_abs, present, config):
    safe_max = max_abs.clamp_min(torch.finfo(torch.float32).tiny)
    exponent = torch.floor(torch.log2(safe_max))
    exp_min = -(1 << (config.shared_exponent_bits - 1))
    exp_max = (1 << (config.shared_exponent_bits - 1)) - 1
    exponent = exponent.clamp(exp_min, exp_max)
    return torch.where(present, exponent, torch.zeros_like(exponent))


def _quantize_blocks(blocks, exponent, config):
    step = torch.pow(2.0, exponent - (config.mantissa_bits - 1))
    mantissa_max = (1 << config.mantissa_bits) - 1
    return torch.round(blocks / step).clamp(-mantissa_max, mantissa_max) * step


def _as_blocks(rows, config):
    width = rows.shape[-1]
    if width % config.block_size:
        raise ValueError("The reduction dimension must be a multiple of the block size.")
    return rows.reshape(-1, width).float().reshape(-1, width // config.block_size, config.block_size)


@torch.no_grad()
def _quantize_bfp_rows(rows, config):
    blocks = _as_blocks(rows, config)
    max_abs = blocks.abs().amax(dim=-1, keepdim=True)
    exponent = _shared_exponent(max_abs, max_abs != 0, config)
    return _quantize_blocks(blocks, exponent, config).reshape(rows.shape).to(rows.dtype)


@torch.no_grad()
def bfp_block_exponent(rows, config):
    blocks = _as_blocks(rows, config)
    max_abs = blocks.abs().amax(dim=-1)
    exponent = _shared_exponent(max_abs, max_abs != 0, config)
    return exponent.to(torch.int32).reshape(*rows.shape[:-1], -1)


@torch.no_grad()
def quantize_bfp(tensor, config, chunk_rows):
    width = tensor.shape[-1]
    flat = tensor.reshape(-1, width)
    output = torch.empty_like(flat)
    for start in range(0, flat.size(0), chunk_rows):
        end = min(start + chunk_rows, flat.size(0))
        output[start:end] = _quantize_bfp_rows(flat[start:end], config)
    return output.reshape_as(tensor)


@torch.no_grad()
def quantize_weight_in_place(weight, config):
    for start in range(0, weight.size(0), config.weight_chunk_rows):
        end = min(start + config.weight_chunk_rows, weight.size(0))
        weight[start:end].copy_(_quantize_bfp_rows(weight[start:end], config))


class BFPLinear(nn.Module):
    def __init__(self, linear, config):
        super().__init__()
        self.linear = linear
        self.config = config

    def forward(self, x):
        x_bfp = quantize_bfp(x, self.config, self.config.activation_chunk_rows)
        return F.linear(x_bfp, self.linear.weight, self.linear.bias).to(torch.float16)


def replace_linear_layers(module, config, prefix=""):
    replaced = []
    for name, child in list(module.named_children()):
        full_name = f"{prefix}.{name}" if prefix else name
        if isinstance(child, nn.Linear):
            if full_name == "lm_head" and not config.quantize_lm_head:
                continue
            quantize_weight_in_place(child.weight, config)
            setattr(module, name, BFPLinear(child, config))
            replaced.append(full_name)
        else:
            replaced.extend(replace_linear_layers(child, config, full_name))
    return replaced


def outlier_threshold(x, outlier_config):
    magnitude = x.float().abs()
    mu = magnitude.mean()
    sigma = magnitude.std(unbiased=False)
    return float((mu + outlier_config.sigma_k * sigma).item())


def _select_largest_candidates(magnitude, candidate, limit):
    if limit is None:
        return candidate
    selected = torch.zeros_like(candidate)
    remaining = candidate.clone()
    positions = torch.arange(magnitude.size(-1), device=magnitude.device)
    positions = positions.view(*([1] * (magnitude.ndim - 1)), magnitude.size(-1))
    for _ in range(limit):
        has_candidate = remaining.any(dim=-1, keepdim=True)
        score = magnitude.masked_fill(~remaining, float("-inf"))
        index = score.argmax(dim=-1, keepdim=True)  # Ties keep the lower K index.
        picked = (positions == index) & has_candidate
        selected = selected | picked
        remaining = remaining & ~picked
    return selected


@torch.no_grad()
def _split_bie_rows(rows, threshold, config, max_outliers):
    blocks = _as_blocks(rows, config)
    magnitude = blocks.abs()
    outlier = _select_largest_candidates(magnitude, magnitude > threshold, max_outliers)
    normal_max = torch.where(outlier, 0.0, magnitude).amax(dim=-1, keepdim=True)
    outlier_max = torch.where(outlier, magnitude, 0.0).amax(dim=-1, keepdim=True)
    normal_exp = _shared_exponent(normal_max, normal_max != 0, config)
    outlier_exp = _shared_exponent(outlier_max, outlier_max != 0, config)
    dequantized = _quantize_blocks(blocks, torch.where(outlier, outlier_exp, normal_exp), config)
    normal_part = torch.where(outlier, 0.0, dequantized).reshape(rows.shape)
    outlier_part = torch.where(outlier, dequantized, 0.0).reshape(rows.shape)
    normal_block_exp = normal_exp.squeeze(-1).to(torch.int32)
    return normal_part, outlier_part, outlier.reshape(rows.shape), normal_block_exp


@torch.no_grad()
def split_bie(tensor, threshold, config, max_outliers, chunk_rows):
    width = tensor.shape[-1]
    flat = tensor.reshape(-1, width)
    normal = torch.empty_like(flat, dtype=torch.float32)
    outlier = torch.empty_like(flat, dtype=torch.float32)
    mask = torch.empty_like(flat, dtype=torch.bool)
    normal_exp = torch.empty(
        (flat.size(0), width // config.block_size), dtype=torch.int32, device=flat.device
    )
    for start in range(0, flat.size(0), chunk_rows):
        end = min(start + chunk_rows, flat.size(0))
        normal[start:end], outlier[start:end], mask[start:end], normal_exp[start:end] = (
            _split_bie_rows(flat[start:end], threshold, config, max_outliers)
        )
    return normal, outlier, mask, normal_exp

## Block-exponent trajectory profiler

`trajectory_stats` takes group partials and their block exponents, both shaped
`(groups, tokens, outputs)` in K order. Partials are exact in FP32 (shared-exponent mantissa
products) and are only used to detect zero partials. `E_acc` is the running maximum of the block
exponents of earlier nonzero partials (`torch.cummax`). Histogram bin `i` holds
`delta_E = i - (L + 1)`; the first and last bins collect `delta_E <= -(L + 1)` and
`delta_E >= L + 1`, which is exact for every window with `T_drop, T_replace <= L`.

In [ ]:
PATH_NAMES = ("raw", "separated")
TRAJECTORY_COUNT_NAMES = ("total_slots", "zero_new", "initial_or_zero_acc_load", "nonzero_decisions")
OUTLIER_COUNT_NAMES = (
    "outlier_partials_nonzero", "outlier_partial_slots",
    "outlier_lanes", "total_lanes", "outlier_groups", "total_groups",
)
TC = {name: index for index, name in enumerate(TRAJECTORY_COUNT_NAMES)}
OC = {name: index for index, name in enumerate(OUTLIER_COUNT_NAMES)}


def num_delta_bins(limit):
    return 2 * limit + 3


def delta_values(limit):
    return torch.arange(num_delta_bins(limit), dtype=torch.int64) - (limit + 1)


NO_EXPONENT = -(1 << 20)  # E_acc before the first nonzero partial (accumulator empty).


@torch.no_grad()
def trajectory_stats(partials, block_exp, limit):
    if partials.shape != block_exp.shape:
        raise ValueError("Partials and block exponents must have the same shape.")
    new_nonzero = partials != 0
    masked = torch.where(new_nonzero, block_exp.to(torch.int32), NO_EXPONENT)
    running_max = torch.cummax(masked, dim=0).values
    del masked
    e_acc = torch.cat((torch.full_like(running_max[:1], NO_EXPONENT), running_max[:-1]), dim=0)
    del running_max
    has_acc = e_acc != NO_EXPONENT
    decision = new_nonzero & has_acc
    delta = (block_exp.to(torch.int32) - e_acc).clamp_(-(limit + 1), limit + 1).add_(limit + 1)
    del e_acc
    sentinel = num_delta_bins(limit)
    delta = torch.where(decision, delta, torch.full_like(delta, sentinel))
    hist = torch.bincount(delta.reshape(-1), minlength=sentinel + 1)[:sentinel].to(torch.int64)
    counts = torch.stack((
        torch.tensor(partials.numel(), dtype=torch.int64, device=partials.device),
        (~new_nonzero).sum(dtype=torch.int64),
        (new_nonzero & ~has_acc).sum(dtype=torch.int64),
        decision.sum(dtype=torch.int64),
    ))
    return hist, counts


def window_counts(hist, limit, t_drop, t_replace):
    if not (1 <= t_drop <= limit and 1 <= t_replace <= limit):
        raise ValueError("Window thresholds must lie in [1, delta_bin_limit].")
    hist = torch.as_tensor(hist, dtype=torch.int64)
    deltas = delta_values(limit)
    drop = int(hist[deltas <= -t_drop].sum().item())
    replace = int(hist[deltas >= t_replace].sum().item())
    return drop, replace, int(hist.sum().item()) - drop - replace


def _rate(numerator, denominator):
    return {
        "numerator": int(numerator),
        "denominator": int(denominator),
        "rate": None if denominator == 0 else numerator / denominator,
    }


def window_row(scope, limit, t_drop, t_replace):
    raw_hist = scope["raw"]["delta_hist"]
    sep_hist = scope["separated"]["delta_hist"]
    raw_decisions = scope["raw"]["counts"]["nonzero_decisions"]
    normal_decisions = scope["separated"]["counts"]["nonzero_decisions"]
    outlier_partials = scope["outlier"]["outlier_partials_nonzero"]

    raw_drop, raw_replace, raw_add = window_counts(raw_hist, limit, t_drop, t_replace)
    sep_drop, sep_replace, sep_add = window_counts(sep_hist, limit, t_drop, t_replace)
    sep_total = normal_decisions + outlier_partials
    raw_oow = _rate(raw_drop + raw_replace, raw_decisions)["rate"]
    sep_oow = _rate(sep_drop + sep_replace, sep_total)["rate"]
    return {
        "t_drop": t_drop,
        "t_replace": t_replace,
        "in_window_delta_range": [-(t_drop - 1), t_replace - 1],
        "window_width": t_drop + t_replace - 1,
        "raw": {
            "in_window": _rate(raw_add, raw_decisions),
            "drop": _rate(raw_drop, raw_decisions),
            "replace": _rate(raw_replace, raw_decisions),
        },
        "separated": {
            "in_window": _rate(sep_add + outlier_partials, sep_total),
            "drop": _rate(sep_drop, sep_total),
            "replace": _rate(sep_replace, sep_total),
            "normal_only_in_window": _rate(sep_add, normal_decisions),
            "outlier_partial_share": _rate(outlier_partials, sep_total),
        },
        "relative_oow_reduction": (
            None if not raw_oow or sep_oow is None else 1.0 - sep_oow / raw_oow
        ),
    }


def evenly_spaced_indices(total, count, device=None):
    if count >= total:
        return torch.arange(total, device=device, dtype=torch.long)
    if count == 1:
        return torch.tensor([total // 2], device=device, dtype=torch.long)
    return torch.linspace(0, total - 1, steps=count, device=device).round().to(torch.long)


def _parse_layer_meta(name):
    match = re.search(r"model\.layers\.(\d+)\.", name)
    return name.rsplit(".", 1)[-1], int(match.group(1)) if match else -1


class InWindowProfiler:
    def __init__(self, bfp_config, outlier_config, profile_config):
        self.bfp = bfp_config
        self.outlier = outlier_config
        self.profile = profile_config
        self.limit = profile_config.delta_bin_limit
        self.layers = {}
        self.handles = []

    def _ensure(self, name, shape_meta):
        if name not in self.layers:
            bins = num_delta_bins(self.limit)
            self.layers[name] = {
                "shape_meta": shape_meta,
                "num_calls": 0,
                "profiled_tokens": 0,
                "thresholds": [],
                "hist": {path: torch.zeros(bins, dtype=torch.int64) for path in PATH_NAMES},
                "counts": {
                    path: torch.zeros(len(TRAJECTORY_COUNT_NAMES), dtype=torch.int64)
                    for path in PATH_NAMES
                },
                "outlier": torch.zeros(len(OUTLIER_COUNT_NAMES), dtype=torch.int64),
            }
        record = self.layers[name]
        if record["shape_meta"] != shape_meta:
            raise RuntimeError(f"Layer shape changed while profiling {name}.")
        return record

    @torch.no_grad()
    def _update(self, name, x, weight):
        bfp, profile = self.bfp, self.profile
        width = x.shape[-1]
        flat = x.reshape(-1, width)
        threshold = outlier_threshold(flat, self.outlier)  # Tensor-wise, over every token of the call.
        rows = flat[::profile.token_stride]

        raw_x = quantize_bfp(rows, bfp, bfp.activation_chunk_rows).float()
        raw_exp = bfp_block_exponent(rows, bfp)
        normal_x, outlier_x, outlier_mask, normal_exp = split_bie(
            rows, threshold, bfp, self.outlier.max_outliers_per_block, bfp.activation_chunk_rows
        )

        total_out = weight.shape[0]
        weight_view = weight.detach()
        if profile.out_subsample is not None and total_out > profile.out_subsample:
            indices = evenly_spaced_indices(total_out, profile.out_subsample, weight_view.device)
            weight_view = weight_view.index_select(0, indices)
        weight_view = weight_view.float()
        weight_exp = bfp_block_exponent(weight_view, bfp)  # Equals the exponent used to quantize W.

        num_tokens = rows.shape[0]
        num_groups = width // bfp.block_size
        profiled_out = weight_view.shape[0]
        record = self._ensure(name, (width, total_out, profiled_out, num_groups))
        record["num_calls"] += 1
        record["profiled_tokens"] += num_tokens
        record["thresholds"].append(threshold)

        device = rows.device
        hist = {path: torch.zeros(num_delta_bins(self.limit), dtype=torch.int64, device=device)
                for path in PATH_NAMES}
        counts = {path: torch.zeros(len(TRAJECTORY_COUNT_NAMES), dtype=torch.int64, device=device)
                  for path in PATH_NAMES}
        outlier_nonzero = torch.zeros((), dtype=torch.int64, device=device)

        def group_major(values):
            return values.reshape(num_tokens, num_groups, bfp.block_size).permute(1, 0, 2).contiguous()

        grouped = {"raw": group_major(raw_x), "normal": group_major(normal_x), "outlier": group_major(outlier_x)}
        del raw_x, normal_x, outlier_x
        grouped_weight = weight_view.reshape(profiled_out, num_groups, bfp.block_size).permute(1, 2, 0).contiguous()
        activation_exp = {"raw": raw_exp.t().contiguous(), "normal": normal_exp.t().contiguous()}  # (G, T)
        weight_exp = weight_exp.t().contiguous()  # (G, O)

        token_chunk = min(profile.token_chunk, num_tokens)
        out_chunk = max(1, profile.max_chunk_elements // (num_groups * token_chunk))
        for t0 in range(0, num_tokens, token_chunk):
            t1 = min(t0 + token_chunk, num_tokens)
            for o0 in range(0, profiled_out, out_chunk):
                o1 = min(o0 + out_chunk, profiled_out)
                w = grouped_weight[:, :, o0:o1]
                for path, source in (("raw", "raw"), ("separated", "normal")):
                    partials = torch.bmm(grouped[source][:, t0:t1], w)
                    block_exp = activation_exp[source][:, t0:t1, None] + weight_exp[:, None, o0:o1]
                    h, c = trajectory_stats(partials, block_exp, self.limit)
                    hist[path] += h
                    counts[path] += c
                    del partials, block_exp
                outlier_partials = torch.bmm(grouped["outlier"][:, t0:t1], w)
                outlier_nonzero += (outlier_partials != 0).sum(dtype=torch.int64)
                del outlier_partials

        group_has_outlier = outlier_mask.reshape(num_tokens, num_groups, bfp.block_size).any(dim=-1)
        slots = num_groups * num_tokens * profiled_out
        outlier_counts = torch.tensor([
            int(outlier_nonzero.item()), slots,
            int(outlier_mask.sum().item()), outlier_mask.numel(),
            int(group_has_outlier.sum().item()), group_has_outlier.numel(),
        ], dtype=torch.int64)
        for path in PATH_NAMES:
            record["hist"][path] += hist[path].cpu()
            record["counts"][path] += counts[path].cpu()
        record["outlier"] += outlier_counts

    def attach(self, model):
        for name, module in model.named_modules():
            if isinstance(module, BFPLinear):
                self.handles.append(module.register_forward_pre_hook(self._make_hook(name)))
        return self

    def _make_hook(self, name):
        def hook(module, args):
            self._update(name, args[0], module.linear.weight)
            return None
        return hook

    def remove(self):
        for handle in self.handles:
            handle.remove()
        self.handles = []

    def _scope(self, hist, counts, outlier):
        return {
            **{
                path: {
                    "counts": {n: int(counts[path][i]) for i, n in enumerate(TRAJECTORY_COUNT_NAMES)},
                    "delta_hist": [int(v) for v in hist[path].tolist()],
                }
                for path in PATH_NAMES
            },
            "outlier": {n: int(outlier[i]) for i, n in enumerate(OUTLIER_COUNT_NAMES)},
        }

    def _windows(self, scope, windows):
        return [window_row(scope, self.limit, t_drop, t_replace) for t_drop, t_replace in windows]

    def export(self):
        grid = [(d, r) for d in self.profile.t_drop_values for r in self.profile.t_replace_values]
        grid += [w for w in self.profile.highlight_windows if w not in grid]
        bins = num_delta_bins(self.limit)
        total_hist = {path: torch.zeros(bins, dtype=torch.int64) for path in PATH_NAMES}
        total_counts = {path: torch.zeros(len(TRAJECTORY_COUNT_NAMES), dtype=torch.int64)
                        for path in PATH_NAMES}
        total_outlier = torch.zeros(len(OUTLIER_COUNT_NAMES), dtype=torch.int64)

        records = []
        for name, record in self.layers.items():
            input_features, total_out, profiled_out, num_groups = record["shape_meta"]
            layer_type, layer_index = _parse_layer_meta(name)
            scope = self._scope(record["hist"], record["counts"], record["outlier"])
            thresholds = record["thresholds"]
            records.append({
                "layer_name": name,
                "layer_type": layer_type,
                "layer_index": layer_index,
                "input_features": input_features,
                "total_output_channels": total_out,
                "profiled_output_channels": profiled_out,
                "num_groups": num_groups,
                "num_calls": record["num_calls"],
                "profiled_tokens": record["profiled_tokens"],
                "outlier_threshold": {
                    "per_call": thresholds,
                    "mean": sum(thresholds) / len(thresholds),
                },
                **scope,
                "windows": self._windows(scope, self.profile.highlight_windows),
            })
            for path in PATH_NAMES:
                total_hist[path] += record["hist"][path]
                total_counts[path] += record["counts"][path]
            total_outlier += record["outlier"]

        records.sort(key=lambda row: (row["layer_index"] < 0, row["layer_index"], row["layer_type"]))
        aggregate = {"num_layers": len(records), **self._scope(total_hist, total_counts, total_outlier)}
        aggregate["windows"] = self._windows(aggregate, grid)
        return aggregate, records


def validate_scope(scope, label):
    raw = scope["raw"]["counts"]
    for path in PATH_NAMES:
        counts = scope[path]["counts"]
        if counts["total_slots"] != (
            counts["zero_new"] + counts["initial_or_zero_acc_load"] + counts["nonzero_decisions"]
        ):
            raise RuntimeError(f"Slot accounting mismatch: {label}/{path}.")
        if sum(scope[path]["delta_hist"]) != counts["nonzero_decisions"]:
            raise RuntimeError(f"Histogram does not cover every decision: {label}/{path}.")
        if counts["total_slots"] != raw["total_slots"]:
            raise RuntimeError(f"Raw and separated slots differ: {label}.")
    outlier = scope["outlier"]
    if outlier["outlier_partial_slots"] != raw["total_slots"]:
        raise RuntimeError(f"Outlier partial slots differ from trajectory slots: {label}.")
    for numerator, denominator in (
        ("outlier_partials_nonzero", "outlier_partial_slots"),
        ("outlier_lanes", "total_lanes"),
        ("outlier_groups", "total_groups"),
    ):
        if outlier[numerator] > outlier[denominator]:
            raise RuntimeError(f"{numerator} exceeds {denominator}: {label}.")
    for row in scope.get("windows", []):
        for path in PATH_NAMES:
            parts = row[path]
            if parts["in_window"]["numerator"] + parts["drop"]["numerator"] \
                    + parts["replace"]["numerator"] != parts["in_window"]["denominator"]:
                raise RuntimeError(f"Window partition mismatch: {label}/{path}.")


def validate_export(aggregate, records):
    validate_scope(aggregate, "aggregate")
    for record in records:
        validate_scope(record, record["layer_name"])

## Synthetic correctness checks

Hand-computed DEWA decisions (including a zero partial that must not move `E_acc`), an
independent Python reference for the running-max histogram, the block-exponent semantics of
raw and BiE-normal partials, and the hook accounting on a toy Linear with one outlier.

In [ ]:
_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_L = PROFILE.delta_bin_limit

# 1. Hand-computed trajectory: six partials of one dot product in K order.
#    t0 load; t1 delta=-2; t2 zero partial with a large exponent (hold, must not raise E_acc);
#    t3 delta=-11; t4 delta=+5 (new maximum -5); t5 delta=-2 against the new maximum.
_p = torch.tensor([700.0, 3.0, 0.0, 5.0, 1.0, 2.0], device=_device).reshape(6, 1, 1)
_e = torch.tensor([-10, -12, -3, -21, -5, -7], device=_device, dtype=torch.int32).reshape(6, 1, 1)
_h, _c = trajectory_stats(_p, _e, _L)
assert _c.tolist() == [6, 1, 1, 4]
_d = delta_values(_L)
assert sorted(int(_d[i]) for i in torch.nonzero(_h.cpu()).flatten()) == [-11, -2, 5]
assert int(_h.cpu()[-2 + _L + 1]) == 2
assert window_counts(_h.cpu(), _L, 9, 3) == (1, 1, 2)    # -11 drop, +5 replace, -2/-2 in-window
assert window_counts(_h.cpu(), _L, 12, 6) == (0, 0, 4)

# 2. Independent Python reference: running max over nonzero partials, zeros, and both edge bins.
torch.manual_seed(0)
_vals = torch.randint(-3, 4, (40, 3, 5)).float()          # about 1/7 zero partials
_exps = torch.randint(-32, 31, (40, 3, 5), dtype=torch.int32)
_vals[:, 0, 0] = torch.tensor([0.0, 0.0] + [1.0] * 38)    # leading zeros before the load
_vals[-1, 1, 2], _exps[-1, 1, 2] = 1.0, 200               # overflow edge bin
_vals[-1, 2, 3], _exps[-1, 2, 3] = 1.0, -200              # underflow edge bin
_h, _c = trajectory_stats(_vals.to(_device), _exps.to(_device), _L)
_ref_h = torch.zeros(num_delta_bins(_L), dtype=torch.int64)
_ref_c = [0, 0, 0, 0]
for _t in range(_vals.size(1)):
    for _o in range(_vals.size(2)):
        _e_acc = None
        for _g in range(_vals.size(0)):
            _ref_c[0] += 1
            if _vals[_g, _t, _o] == 0:
                _ref_c[1] += 1
                continue
            _e_new = int(_exps[_g, _t, _o])
            if _e_acc is None:
                _ref_c[2] += 1
                _e_acc = _e_new
                continue
            _ref_c[3] += 1
            _ref_h[max(-(_L + 1), min(_L + 1, _e_new - _e_acc)) + _L + 1] += 1
            _e_acc = max(_e_acc, _e_new)
assert _c.tolist() == _ref_c, (_c.tolist(), _ref_c)
assert torch.equal(_h.cpu(), _ref_h)
assert _ref_h[0] > 0 and _ref_h[-1] > 0

# 3. Block-exponent semantics: P = Psum * 2^(E_w + E_a - 2(M - 1)) with an integer Psum and
#    |Psum| <= block * (2^M - 1)^2, for raw BFP lanes and for the BiE normal lanes.
_x = torch.randn(6, 64, device=_device, dtype=torch.float16)
_x[0, 3] = 900.0
_x[2, 40] = -300.0
_x[4, 16:32] = 0.0                                        # all-zero activation block
_w = torch.randn(7, 64, device=_device, dtype=torch.float16) * 0.02
_w_exp = bfp_block_exponent(_w, BFP)
quantize_weight_in_place(_w, BFP)
assert torch.equal(bfp_block_exponent(_w, BFP), _w_exp)  # the exponent survives quantization
_thr = outlier_threshold(_x, OUTLIER)
_n, _o, _m, _n_exp = split_bie(_x, _thr, BFP, None, 4)
assert torch.equal(_m, _x.float().abs() > _thr) and bool(_m[0, 3]) and bool(_m[2, 40])
assert torch.all(_n[_m] == 0) and torch.all(_o[~_m] == 0)
_raw = quantize_bfp(_x, BFP, 4).float()
_raw_exp = bfp_block_exponent(_x, BFP)
assert (_x.float() - (_n + _o))[0].abs().sum() < (_x.float() - _raw)[0].abs().sum()
assert int(_n_exp[0, 0]) < int(_raw_exp[0, 0])            # the outlier no longer sets E_a
_gw = _w.float().reshape(7, 4, 16).permute(1, 2, 0).contiguous()
_lsb = 2 * (BFP.mantissa_bits - 1)
_psum_max = BFP.block_size * ((1 << BFP.mantissa_bits) - 1) ** 2
for _xq, _xe in ((_raw, _raw_exp), (_n, _n_exp)):
    _gx = _xq.reshape(6, 4, 16).permute(1, 0, 2).contiguous()
    _part = torch.bmm(_gx, _gw).double()
    assert torch.equal(_part, torch.bmm(_gx.double(), _gw.double()))   # FP32 partials are exact
    _shift = (_xe.t()[:, :, None] + _w_exp.t()[:, None, :] - _lsb).cpu()
    _psum = torch.ldexp(_part.cpu(), -_shift)                # exact power-of-two scaling on CPU
    assert torch.equal(_psum, _psum.round()) and _psum.abs().max() <= _psum_max
_, _, _m_top1, _ = split_bie(_x, 0.0, BFP, 1, 4)
assert int(_m_top1.reshape(6, 4, 16).sum(-1).max()) == 1

# 4. Hook accounting on a toy Linear with one outlier, small chunks to exercise tiling.
_toy_profile = ProfileConfig(
    max_profile_blocks=1, token_stride=1, out_subsample=None,
    token_chunk=2, max_chunk_elements=24, delta_bin_limit=_L,
)
_lin = nn.Linear(64, 7, bias=False, device=_device, dtype=torch.float16)
quantize_weight_in_place(_lin.weight, BFP)
_inp = torch.randn(1, 5, 64, device=_device, dtype=torch.float16)
_inp[0, 1, 9] = 1000.0
_prof = InWindowProfiler(BFP, OUTLIER, _toy_profile)
_prof._update("model.layers.0.self_attn.q_proj", _inp, _lin.weight)
_agg, _recs = _prof.export()
validate_export(_agg, _recs)
assert _agg["raw"]["counts"]["total_slots"] == 5 * 7 * 4
assert _agg["outlier"]["outlier_lanes"] == 1 and _agg["outlier"]["outlier_groups"] == 1
assert _agg["outlier"]["outlier_partials_nonzero"] == int((_lin.weight.float()[:, 9] != 0).sum())
_rows = _inp.reshape(-1, 64)
_gx = quantize_bfp(_rows, BFP, 4).float().reshape(5, 4, 16).permute(1, 0, 2)
_gw = _lin.weight.float().reshape(7, 4, 16).permute(1, 2, 0)
_be = bfp_block_exponent(_rows, BFP).t()[:, :, None] + bfp_block_exponent(_lin.weight, BFP).t()[:, None, :]
_h_direct, _ = trajectory_stats(torch.bmm(_gx, _gw), _be, _L)
assert _agg["raw"]["delta_hist"] == _h_direct.cpu().tolist()   # tiling does not change the result
_full = _prof._windows(_agg, [(9, 3)])[0]["separated"]["in_window"]
assert _full["denominator"] == (_agg["separated"]["counts"]["nonzero_decisions"]
                                + _agg["outlier"]["outlier_partials_nonzero"])
del _prof, _agg, _recs, _lin
print("Synthetic block-exponent trajectory and in-window checks passed.")

## Load tokenizer and sampled WikiText-2 blocks

Complete, non-overlapping 2048-token blocks sampled uniformly across the test split, as in the
earlier profilers. No labels, loss, or PPL are computed.

In [ ]:
token = os.getenv("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
if not token:
    token = getpass("HF_TOKEN: ")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=token)
dataset = load_dataset(DATASET_ID, DATASET_CONFIG, split=SPLIT)
text = "\n\n".join(dataset["text"])
input_ids = tokenizer(text, return_tensors="pt").input_ids
usable_tokens = input_ids.size(1) // CONTEXT_LENGTH * CONTEXT_LENGTH
total_blocks = usable_tokens // CONTEXT_LENGTH
profile_blocks = min(PROFILE.max_profile_blocks, total_blocks)
profile_block_indices = evenly_spaced_indices(total_blocks, profile_blocks).tolist()
del dataset, text

print(
    f"WikiText-2 {SPLIT}: {input_ids.numel():,} tokens, "
    f"{total_blocks} complete blocks, sampled blocks={profile_block_indices}"
)

## Run the paired raw / separated trajectory profiling

The model executes the raw W-BFP4 / A-BFP4 path only to produce common downstream activations.
The hook replays both shadow paths and stores per-layer and aggregate histograms.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("The full profiler requires an NVIDIA CUDA GPU.")

torch.manual_seed(0)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map=0,
    attn_implementation="eager",
    token=token,
)
model.eval()
model.config.use_cache = False
quantized_layers = replace_linear_layers(model, BFP)
if len(quantized_layers) != 224:
    raise RuntimeError(f"Expected 224 decoder Linear layers, found {len(quantized_layers)}.")
torch.cuda.empty_cache()

_tokens_per_call = math.ceil(CONTEXT_LENGTH / PROFILE.token_stride)
_slots = 0
for _name, _module in model.named_modules():
    if isinstance(_module, BFPLinear):
        _out = _module.linear.out_features
        _out = min(_out, PROFILE.out_subsample) if PROFILE.out_subsample else _out
        _slots += _tokens_per_call * _out * (_module.linear.in_features // BFP.block_size)
print(f"Group-output slots per path: {_slots * profile_blocks:,}")

profiler = InWindowProfiler(BFP, OUTLIER, PROFILE).attach(model)
device = next(model.parameters()).device
torch.cuda.reset_peak_memory_stats(device)
start = time.perf_counter()
try:
    with torch.inference_mode():
        for block_index in tqdm(profile_block_indices, desc="Profiling DEWA in-window rate"):
            begin = block_index * CONTEXT_LENGTH
            batch = input_ids[:, begin:begin + CONTEXT_LENGTH].to(device)
            model(batch, use_cache=False)
finally:
    profiler.remove()
torch.cuda.synchronize(device)
elapsed = time.perf_counter() - start
peak_memory_gib = torch.cuda.max_memory_allocated(device) / (1024 ** 3)

aggregate, records = profiler.export()
validate_export(aggregate, records)
if len(records) != len(quantized_layers):
    raise RuntimeError("Not every quantized Linear layer was profiled.")
if any(record["num_calls"] != profile_blocks for record in records):
    raise RuntimeError("Unexpected per-layer profiling call count.")

payload = {
    "metadata": {
        "model": MODEL_ID,
        "dataset": f"{DATASET_ID}/{DATASET_CONFIG}",
        "split": SPLIT,
        "analysis": "DEWA in-window rate with and without activation-outlier separation",
        "ppl_evaluated": False,
        "bfp_config": asdict(BFP),
        "outlier_config": asdict(OUTLIER),
        "profile_config": asdict(PROFILE),
        "forward_path": "W-BFP4 / A-BFP4 / G16 on every decoder Linear; lm_head FP16",
        "raw_path": "activation BFP4 (one shared exponent per G16 block); every group partial enters the accumulator",
        "separated_path": "activation BiE4 with tensor-wise |x| > mu + k*sigma outliers; P_N enters the accumulator, nonzero P_O counts as in-window (routed to FP-Acc)",
        "trajectory": "group partials in true K order; decisions depend only on block exponents, so any window is derived afterwards from one replay",
        "delta_e": "E_t - E_acc(t); E_t = E_w + E_a block exponent (normal E_a on the separated path); E_acc(t) = max block exponent of earlier nonzero partials on the same path (DEW-ACC exponent register)",
        "window_rule": "drop if delta<=-T_drop; replace if delta>=T_replace; otherwise in-window",
        "load_rule": "zero partial: hold (no decision, E_acc unchanged); first nonzero partial: load, excluded from rates",
        "not_modeled": "accumulator-width overflow spills (they reset E_acc but do not depend on the window)",
        "raw_in_window_rate": "in-window decisions / nonzero decisions",
        "separated_in_window_rate": "(normal in-window decisions + nonzero P_O) / (normal nonzero decisions + nonzero P_O)",
        "delta_hist_bins": f"bin i -> delta_E = i - {PROFILE.delta_bin_limit + 1}; first/last bins are <= / >= edges",
        "evaluation_protocol": EVALUATION_PROTOCOL,
        "context_length": CONTEXT_LENGTH,
        "source_input_tokens": int(input_ids.numel()),
        "complete_blocks": total_blocks,
        "profile_blocks": profile_blocks,
        "profile_block_indices": profile_block_indices,
        "profiled_tokens_per_block": _tokens_per_call,
        "output_channel_sampling": "all_if_within_cap_else_evenly_spaced_inclusive",
        "quantized_linear_layers": len(quantized_layers),
        "elapsed_seconds": elapsed,
        "peak_gpu_memory_gib": peak_memory_gib,
        "gpu": torch.cuda.get_device_name(0),
        "cuda": torch.version.cuda,
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
    },
    "aggregate": aggregate,
    "layers": records,
}

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved: {OUTPUT_PATH.resolve()}")
print(f"Layers={len(records)}, elapsed={elapsed:.1f}s, peak GPU={peak_memory_gib:.2f} GiB")

## Summary

Highlighted windows first (`(9, 3)` is the current asymmetric design point), then the in-window
grid, then the shape of the `delta_E` distribution on each path.

In [ ]:
def _pct(rate):
    return "   n/a" if rate is None else f"{rate:9.4%}"


rows = {(row["t_drop"], row["t_replace"]): row for row in aggregate["windows"]}
print("(T_drop,T_replace)  in-window dE   width   raw in-win  sep in-win  normal-only  rel. OOW red.")
for key in PROFILE.highlight_windows:
    row = rows[key]
    low, high = row["in_window_delta_range"]
    reduction = row["relative_oow_reduction"]
    reduction_text = "n/a" if reduction is None else f"{reduction:.2%}"
    print(
        f"{str(key):>18}  [{low:>3}, {high:>2}]  {row['window_width']:>6}  "
        f"{_pct(row['raw']['in_window']['rate'])}  {_pct(row['separated']['in_window']['rate'])}  "
        f"{_pct(row['separated']['normal_only_in_window']['rate'])}  {reduction_text:>12}"
    )

for path in PATH_NAMES:
    print(f"\n{path} in-window rate (rows: T_drop, columns: T_replace)")
    print("      " + "".join(f"{r:>9}" for r in PROFILE.t_replace_values))
    for t_drop in PROFILE.t_drop_values:
        cells = "".join(
            f"{rows[(t_drop, r)][path]['in_window']['rate'] or 0:9.4%}" for r in PROFILE.t_replace_values
        )
        print(f"{t_drop:>5} {cells}")

_deltas = delta_values(PROFILE.delta_bin_limit)
for path in PATH_NAMES:
    hist = torch.tensor(aggregate[path]["delta_hist"], dtype=torch.float64)
    total = hist.sum().item()
    if not total:
        continue
    negative = hist[_deltas < 0].sum().item() / total
    zero = hist[_deltas == 0].sum().item() / total
    positive = hist[_deltas > 0].sum().item() / total
    cdf = (hist.cumsum(0) / total).tolist()
    pick = lambda q: int(_deltas[next(i for i, v in enumerate(cdf) if v >= q)])
    print(
        f"\n{path} delta_E over nonzero decisions: <0 {negative:.4%}, =0 {zero:.4%}, >0 {positive:.4%}; "
        f"P1={pick(0.01)}, P5={pick(0.05)}, P50={pick(0.5)}, P95={pick(0.95)}, P99={pick(0.99)}"
    )

outlier = aggregate["outlier"]
print(
    f"\nOutlier lanes: {outlier['outlier_lanes']:,}/{outlier['total_lanes']:,} "
    f"({outlier['outlier_lanes'] / outlier['total_lanes']:.4%}); "
    f"affected G{BFP.block_size} groups: {outlier['outlier_groups'] / outlier['total_groups']:.4%}; "
    f"nonzero P_O: {outlier['outlier_partials_nonzero'] / outlier['outlier_partial_slots']:.4%} of slots"
)

In [ ]:
if not OUTPUT_PATH.is_file():
    raise FileNotFoundError(f"Profiler output does not exist: {OUTPUT_PATH}")

try:
    from google.colab import files
except ImportError:
    print(f"Not running in Colab. JSON remains at: {OUTPUT_PATH.resolve()}")
else:
    files.download(str(OUTPUT_PATH))